# Phase 6 — Disfluency Detection Training
**Model:** Wav2Vec2-base fine-tuned for stuttering detection (5 classes)

**Dataset:** SEP-28k (Apple's stuttering events dataset)

**Runtime:** Set to **T4 GPU** before running: Runtime → Change runtime type → T4 GPU

**Time:** Download ~30-60 min, Training ~4 hours on T4

## Step 1: Setup & GPU Check

In [ ]:
!pip install -q transformers datasets soundfile accelerate librosa
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone SEP-28k Repository

In [ ]:
import os

if not os.path.exists('/content/ml-stuttering-events-dataset'):
    !git clone https://github.com/apple/ml-stuttering-events-dataset.git
else:
    print('Repository already cloned')

!ls /content/ml-stuttering-events-dataset/

## Step 3: Download Raw Podcast Audio
Downloads ~32 GB of raw podcast WAV files. Takes 30-60 min.
If you've already downloaded them to Drive before, they'll be reused.

In [ ]:
RAW_WAV_DIR = '/content/raw_wavs'
CLIPS_DIR = '/content/SEP-28k_clips'
DRIVE_CLIPS = '/content/drive/MyDrive/SEP-28k_clips'

# Check if clips already exist on Drive from a previous run
if os.path.exists(DRIVE_CLIPS) and len(os.listdir(DRIVE_CLIPS)) > 1000:
    CLIPS_DIR = DRIVE_CLIPS
    print(f'Using cached clips from Drive ({len(os.listdir(CLIPS_DIR))} files)')
    print('Skipping download — jump to Step 5')
else:
    print('Downloading raw podcast audio (~32 GB)...')
    print('This will take 30-60 minutes. Keep this tab active.')
    os.makedirs(RAW_WAV_DIR, exist_ok=True)
    %cd /content/ml-stuttering-events-dataset
    !pip install -q requests
    !python download_audio.py --episodes SEP-28k_episodes.csv --wavs {RAW_WAV_DIR}
    %cd /content
    print('\nDownload complete.')

## Step 4: Extract 3-Second Clips
Extracts ~28,000 clips from the raw audio (~2.6 GB). Takes ~10 min.

In [ ]:
# Skip if clips already loaded from Drive
if os.path.exists(CLIPS_DIR) and len(os.listdir(CLIPS_DIR)) > 1000:
    print(f'Clips already ready: {len(os.listdir(CLIPS_DIR))} files in {CLIPS_DIR}')
else:
    print('Extracting 3-second clips...')
    os.makedirs(CLIPS_DIR, exist_ok=True)
    %cd /content/ml-stuttering-events-dataset
    !python extract_clips.py --labels SEP-28k_labels.csv --wavs {RAW_WAV_DIR} --clips {CLIPS_DIR}
    %cd /content

    clip_count = len(os.listdir(CLIPS_DIR))
    print(f'\nExtracted {clip_count} clips to {CLIPS_DIR}')

    # Cache clips to Drive for future runs
    print('Saving clips to Google Drive (so you don\'t have to re-download)...')
    !cp -r {CLIPS_DIR} {DRIVE_CLIPS}
    print('Saved to Drive.')

## Step 5: Inspect Dataset
Check the CSV columns and clip filenames so parsing works correctly.

In [ ]:
import pandas as pd

LABELS_CSV = '/content/ml-stuttering-events-dataset/SEP-28k_labels.csv'

df = pd.read_csv(LABELS_CSV)
print('Columns:', df.columns.tolist())
print(f'Total rows: {len(df)}')
print()
df.head(10)

In [ ]:
# Check what clip filenames look like
clip_files = os.listdir(CLIPS_DIR)[:10]
print('Sample clip filenames:')
for f in sorted(clip_files):
    print(f'  {f}')
print(f'\nTotal clips: {len(os.listdir(CLIPS_DIR))}')

## Step 6: Model & Dataset Code
All code in one cell — self-contained. The parser auto-detects CSV column names.

In [ ]:
import csv
import random
import logging
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from sklearn.metrics import f1_score, classification_report
from collections import Counter
import librosa

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
logger = logging.getLogger(__name__)

# Our 5 target classes
DISFLUENCY_CLASSES = ['Fluent', 'Repetition', 'Prolongation', 'Block', 'Interjection']
CLASS_WEIGHTS = [1.0, 6.5, 8.0, 12.0, 5.5]

# Mapping from possible SEP-28k CSV column names to our class indices
# Sound Repetition + Word Repetition -> Repetition (index 1)
COLUMN_TO_CLASS = {
    'Prolongation': 2,
    'Block': 3,
    'Interjection': 4,
    'SoundRepetition': 1,
    'Sound Repetition': 1,
    'WordRepetition': 1,
    'Word Repetition': 1,
    'Repetition': 1,
}


class DisfluencyDetector(nn.Module):
    def __init__(self, num_classes=5, freeze_layers=3):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        self.encoder.feature_extractor._freeze_parameters()
        for i in range(freeze_layers):
            for param in self.encoder.encoder.layers[i].parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, input_values):
        outputs = self.encoder(input_values)
        hidden = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(hidden)


class DisfluencyDataset(Dataset):
    def __init__(self, samples, audio_dir, sr=16000, max_len_sec=3.0, augment=False):
        self.samples = samples
        self.audio_dir = audio_dir
        self.sr = sr
        self.max_len = int(max_len_sec * sr)
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        audio_path = os.path.join(self.audio_dir, sample['file'])
        try:
            audio, _ = librosa.load(audio_path, sr=self.sr)
        except Exception:
            audio = np.zeros(self.max_len, dtype=np.float32)

        if self.augment:
            audio = self._augment(audio)

        if len(audio) > self.max_len:
            audio = audio[:self.max_len]
        elif len(audio) < self.max_len:
            audio = np.pad(audio, (0, self.max_len - len(audio)))

        return torch.tensor(audio, dtype=torch.float32), sample['label']

    def _augment(self, audio):
        if random.random() < 0.3:
            rate = random.uniform(0.9, 1.1)
            audio = librosa.effects.time_stretch(audio, rate=rate)
        if random.random() < 0.3:
            noise = np.random.normal(0, 0.005, len(audio)).astype(np.float32)
            audio = audio + noise
        if random.random() < 0.2:
            steps = random.uniform(-1, 1)
            audio = librosa.effects.pitch_shift(audio, sr=self.sr, n_steps=steps)
        return audio.astype(np.float32)


def parse_sep28k_labels(labels_csv, clips_dir):
    """Parse SEP-28k_labels.csv with auto-detected column names.

    Maps Sound Repetition + Word Repetition -> Repetition.
    Only keeps samples with >=2 annotator agreement for non-fluent classes.
    Auto-detects clip filename format by checking what exists on disk.
    """
    samples = []
    skipped_no_file = 0

    with open(labels_csv) as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        print(f'CSV columns: {columns}')

        # Find which disfluency columns exist in this CSV
        active_columns = {}
        for col in columns:
            col_stripped = col.strip()
            if col_stripped in COLUMN_TO_CLASS:
                active_columns[col] = COLUMN_TO_CLASS[col_stripped]
        print(f'Disfluency columns found: {active_columns}')

        # Read first row to detect filename format
        rows = list(reader)

    # Try to detect clip filename format from first few rows
    sample_row = rows[0]
    show = sample_row.get('Show', '').strip()
    ep_id = sample_row.get('EpId', '').strip()
    clip_id = sample_row.get('ClipId', '').strip()

    # Try common filename patterns
    candidates = [
        f'{show}_{ep_id}_{clip_id}.wav',
        f'{show}_{ep_id}_clip{clip_id}.wav',
        f'{show}_{ep_id}_{int(clip_id):04d}.wav' if clip_id.isdigit() else None,
        f'{show}_{ep_id}_{int(clip_id):05d}.wav' if clip_id.isdigit() else None,
    ]
    candidates = [c for c in candidates if c is not None]

    detected_format = None
    clip_files_set = set(os.listdir(clips_dir))

    for fmt in candidates:
        if fmt in clip_files_set:
            detected_format = fmt
            break

    if detected_format:
        # Figure out which format string to use
        if detected_format == candidates[0]:
            fmt_func = lambda s, e, c: f"{s}_{e}_{c}.wav"
        elif detected_format == candidates[1]:
            fmt_func = lambda s, e, c: f"{s}_{e}_clip{c}.wav"
        elif len(candidates) > 2 and detected_format == candidates[2]:
            fmt_func = lambda s, e, c: f"{s}_{e}_{int(c):04d}.wav"
        elif len(candidates) > 3 and detected_format == candidates[3]:
            fmt_func = lambda s, e, c: f"{s}_{e}_{int(c):05d}.wav"
        print(f'Detected filename format: {detected_format}')
    else:
        # Fallback: try Show_EpId_ClipId.wav
        fmt_func = lambda s, e, c: f"{s}_{e}_{c}.wav"
        print(f'Could not auto-detect format. Using: Show_EpId_ClipId.wav')
        print(f'First candidate was: {candidates[0]}')
        print(f'Sample clips on disk: {sorted(list(clip_files_set))[:5]}')

    for row in rows:
        show = row.get('Show', '').strip()
        ep_id = row.get('EpId', '').strip()
        clip_id = row.get('ClipId', '').strip()
        filename = fmt_func(show, ep_id, clip_id)

        # Determine label: pick class with highest annotator votes
        label = 0  # Fluent by default
        max_votes = 0
        for col, cls_idx in active_columns.items():
            try:
                votes = int(row.get(col, 0))
            except (ValueError, TypeError):
                votes = 0
            # For Repetition (idx=1), combine Sound+Word repetition votes
            if cls_idx == 1 and votes > 0:
                # Keep the max across repetition columns
                if votes > max_votes:
                    max_votes = votes
                    label = cls_idx
            elif votes > max_votes:
                max_votes = votes
                label = cls_idx

        # Only keep non-fluent samples with >=2 annotator agreement
        if max_votes < 2 and label != 0:
            continue

        # Check file exists
        if filename not in clip_files_set:
            skipped_no_file += 1
            continue

        samples.append({'file': filename, 'label': label})

    print(f'\nParsed {len(samples)} samples (skipped {skipped_no_file} missing files)')
    counts = Counter(s['label'] for s in samples)
    for i, name in enumerate(DISFLUENCY_CLASSES):
        print(f'  {name}: {counts.get(i, 0)}')

    return samples


print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print('Model and dataset code loaded.')

## Step 7: Parse Dataset & Create Splits

In [ ]:
LABELS_CSV = '/content/ml-stuttering-events-dataset/SEP-28k_labels.csv'

samples = parse_sep28k_labels(LABELS_CSV, CLIPS_DIR)

# Shuffle and split 80/10/10
random.seed(42)
random.shuffle(samples)
n = len(samples)
train_samples = samples[:int(0.8 * n)]
val_samples = samples[int(0.8 * n):int(0.9 * n)]
test_samples = samples[int(0.9 * n):]
print(f'\nTrain: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

## Step 8: Train
Takes ~4 hours on T4 GPU. Progress prints every 200 steps.

In [ ]:
# ── Hyperparameters ──
EPOCHS = 15
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR_ENCODER = 2e-5
LR_HEAD = 1e-3
MODEL_DIR = '/content/models/disfluency'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(MODEL_DIR, exist_ok=True)

# Dataloaders
train_ds = DisfluencyDataset(train_samples, CLIPS_DIR, augment=True)
val_ds = DisfluencyDataset(val_samples, CLIPS_DIR)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)

# Model
model = DisfluencyDetector()
model.to(device)

# Loss with class weights
weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# Differential LR
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': LR_ENCODER},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=1e-2)

# Cosine warmup scheduler
total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR_ENCODER, LR_HEAD],
    total_steps=total_steps, pct_start=warmup_steps/total_steps,
    anneal_strategy='cos'
)

scaler = torch.amp.GradScaler()
best_f1 = 0.0

print(f'Training for {EPOCHS} epochs on {device}')
print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, (audio, labels) in enumerate(train_loader):
        audio, labels = audio.to(device), labels.to(device)

        with torch.amp.autocast(device_type='cuda'):
            logits = model(audio)
            loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 200 == 0:
            print(f'  Epoch {epoch} step {step+1}/{len(train_loader)} loss={loss.item()*GRAD_ACCUM:.4f}')

    train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for audio, labels in val_loader:
            audio = audio.to(device)
            with torch.amp.autocast(device_type='cuda'):
                logits = model(audio)
            val_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            val_true.extend(labels.tolist())

    macro_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)
    print(f'Epoch {epoch:2d} | Loss: {train_loss:.4f} | Macro F1: {macro_f1:.4f}')

    # Save checkpoint
    torch.save({
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'macro_f1': macro_f1,
    }, os.path.join(MODEL_DIR, f'checkpoint_epoch{epoch}.pt'))

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'macro_f1': macro_f1,
        }, os.path.join(MODEL_DIR, 'best_model.pt'))
        print(f'  ✓ Saved best model (Macro F1={macro_f1:.4f})')

print(f'\nTraining complete. Best Macro F1: {best_f1:.4f}')

## Step 9: Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(MODEL_DIR, 'best_model.pt'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]}')

# Test
test_ds = DisfluencyDataset(test_samples, CLIPS_DIR)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)

test_preds, test_true = [], []
with torch.no_grad():
    for audio, labels in test_loader:
        audio = audio.to(device)
        with torch.amp.autocast(device_type='cuda'):
            logits = model(audio)
        test_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        test_true.extend(labels.tolist())

test_f1 = f1_score(test_true, test_preds, average='macro', zero_division=0)
print(f'\nTest Macro F1: {test_f1:.4f}')
print(f'Target: >= 0.72')
print(f'Result: {"PASSED" if test_f1 >= 0.72 else "BELOW TARGET"}')
print()
print(classification_report(test_true, test_preds,
                            target_names=DISFLUENCY_CLASSES, zero_division=0))

## Step 10: Save to Google Drive
After this, download `disfluency_best_model.pt` from your Google Drive and place it at:
```
~/Desktop/Claude-assistant/models/disfluency/best_model.pt
```

In [ ]:
# Save best model to Google Drive
DRIVE_MODEL_DIR = '/content/drive/MyDrive/voice_pipeline_models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

!cp {MODEL_DIR}/best_model.pt {DRIVE_MODEL_DIR}/disfluency_best_model.pt

print(f'\nModel saved to Google Drive:')
print(f'  {DRIVE_MODEL_DIR}/disfluency_best_model.pt')
print(f'\nNext: Download this file to your Mac at:')
print(f'  ~/Desktop/Claude-assistant/models/disfluency/best_model.pt')